# ⚡ Módulo 11 - Notebook 03: Transformaciones Lazy y Acciones

## 🐌 El Modelo de evaluación perezosa de Spark

**Libro:** Saliendo de lo Pandito  
**Módulo:** 11 - PySpark Core SparkSession  
**Duración estimada:** 65 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Distinguir** entre transformaciones lazy y acciones  
✅ **Entender** el DAG (Grafo Acíclico Dirigido)  
✅ **Optimizar** pipelines con lazy evaluation  
✅ **Usar** acciones correctamente  
✅ **Evitar** múltiples materializaciones

---

## 📋 Pre-requisitos

* ✅ Notebooks 11_01 y 11_02 completados
* ✅ Conocimiento de Spark DataFrames
* ✅ Familiaridad con transformaciones básicas

---

## 📚 Contenido

1. Lazy Evaluation: ¿Qué y Por Qué?
2. Transformaciones vs Acciones
3. DAG (Directed Acyclic Graph)
4. Catalyst Optimizer
5. Cache y Persist
6. Caso Integrador: Pipeline Optimizado

---

## 💡 Por qué importa

**Lazy evaluation es el corazón de Spark:**

* ⚡ **Optimización:** Spark optimiza TODO el plan
* 📊 **Eficiencia:** Evita cálculos innecesarios
* 🧠 **Inteligencia:** Catalyst Optimizer mejora tu código
* 🚀 **Performance:** 10-100x más rápido que ejecución eaguer

**Entender lazy = dominar Spark**

In [0]:
import pandas as pd
import numpy as np
import time

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    print("\n🐌 DEMOSTRACIÓN: Lazy Evaluation")
    print("   Las siguientes operaciones NO ejecutan nada aún...")
    
    start = time.time()
    
    # Transformaciones (lazy - no ejecutan)
    df1 = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    df2 = df1.filter("ventas > 1000")
    df3 = df2.select("sucursal_nombre", "ventas")
    df4 = df3.groupBy("sucursal_nombre").sum("ventas")
    
    lazy_time = time.time() - start
    
    print(f"\n   ⏱️  Tiempo de 4 transformaciones: {lazy_time:.4f} segundos")
    print("   🐌 ¡Instantáneo! Porque NO ejecutó nada")
    
    # Acción (trigger - ejecuta TODO)
    print("\n   Ahora ejecutamos una ACCIÓN (.count())...")
    start = time.time()
    resultado = df4.count()
    action_time = time.time() - start
    
    print(f"\n✅ Datos reales procesados")
    print(f"   📊 Sucursales únicas: {resultado}")
    print(f"   ⏱️  Tiempo de ejecución: {action_time:.4f} segundos")
    print(f"   ⚡ AHORA sí ejecutó todo el pipeline")
    
    print(f"\n📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    print(f"\n🎯 Este notebook explorará:")
    print(f"   • Diferencia entre transformaciones y acciones")
    print(f"   • Cómo Spark optimiza el DAG")
    print(f"   • Cuándo usar cache/persist")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df4 = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Lazy Evaluation: Optimización Automática

### 🐌 ¿Qué es Lazy Evaluation?

**Lazy Evaluation:** Spark NO ejecuta transformaciones hasta que se le ordena (acción).

**Analogía:**
```
Pandas (Eager):  "Hazlo AHORA"
Spark (Lazy):    "Anota la tarea, la haré después"
```

**Ejemplo:**
```python
df1 = spark.table("ventas")           # No ejecuta
df2 = df1.filter("ventas > 1000")     # No ejecuta
df3 = df2.select("sucursal", "ventas") # No ejecuta
df3.count()                            # ¡AHORA ejecuta todo!
```

---

### 🔄 Transformaciones vs Acciones

**Transformaciones (Lazy):**
* Crean un NUEVO DataFrame
* NO ejecutan nada
* Registran operación en el DAG

**Ejemplos:**
```python
df.select(...)
df.filter(...)
df.groupBy(...)
df.join(...)
df.orderBy(...)
df.withColumn(...)
```

---

**Acciones (Eager):**
* Ejecutan TODO el DAG
* Devuelven resultados al Driver
* Materializan datos

**Ejemplos:**
```python
df.count()          # Cuenta filas
df.show()           # Muestra filas
df.collect()        # Trae TODO a memoria
df.take(n)          # Trae n filas
df.first()          # Trae 1 fila
df.write.parquet()  # Escribe a disco
df.toPandas()       # Convierte a Pandas
```

---

### 🗓️ DAG (Directed Acyclic Graph)

**DAG:** Grafo de operaciones que Spark construye.

**Flujo:**
```
1. Usuario escribe transformaciones (lazy)
2. Spark construye el DAG
3. Catalyst Optimizer mejora el DAG
4. Usuario ejecuta acción
5. Spark ejecuta el DAG optimizado
```

**Visualización del DAG:**
```
                [tabla: ventas]
                      ↓
              [filter: ventas > 1000]
                      ↓
          [select: sucursal, ventas]
                      ↓
            [groupBy: sucursal]
                      ↓
                  [count]
```

---

### 🧠 Catalyst Optimizer

**Catalyst:** Motor de optimización de Spark.

**Optimizaciones automáticas:**

1. **Predicate Pushdown:**
   ```python
   # Tu código
   df = spark.table("ventas").select("col1", "col2").filter("col1 > 100")
   
   # Catalyst reordena
   df = spark.table("ventas").filter("col1 > 100").select("col1", "col2")
   # Filtra ANTES de leer todo
   ```

2. **Column Pruning:**
   ```python
   # Solo lee columnas usadas
   df.select("col1").show()  # No lee otras columnas
   ```

3. **Join Reordering:**
   ```python
   # Catalyst elige el mejor orden de joins
   ```

---

### 💾 Cache y Persist

**Problema:** Si usas el mismo DataFrame varias veces, Spark lo recalcula cada vez.

**Solución:** `.cache()` o `.persist()`

**Sin cache:**
```python
df2 = df1.filter(...)  # Operación costosa
df2.count()            # Ejecuta todo
df2.show()             # Ejecuta TODO OTRA VEZ
```

**Con cache:**
```python
df2 = df1.filter(...).cache()  # Marcar para cachear
df2.count()                     # Ejecuta y cachea
df2.show()                      # USA el cache (instantáneo)
```

**Regla:**
* Usa `.cache()` cuando usas el DataFrame **más de 1 vez**
* Llama `.unpersist()` cuando termines

---

### 💡 Ejemplo Comparativo

**Eager (como Pandas):**
```python
# Cada operación ejecuta inmediatamente
df1 = pd.read_csv("ventas.csv")       # Lee TODO
df2 = df1[df1['ventas'] > 1000]       # Filtra TODO
df3 = df2[['sucursal', 'ventas']]     # Selecciona TODO
result = df3.groupby('sucursal').sum() # Agrupa TODO
```

**Lazy (Spark):**
```python
# Ninguna operación ejecuta hasta count()
df1 = spark.read.csv("ventas.csv")           # No ejecuta
df2 = df1.filter(df1['ventas'] > 1000)       # No ejecuta
df3 = df2.select('sucursal', 'ventas')       # No ejecuta
result = df3.groupBy('sucursal').sum()       # No ejecuta
result.count()  # ¡AHORA ejecuta todo optimizado!
```

---

### ⚡ Ventajas de Lazy Evaluation

✅ **Optimización:** Spark ve TODO el plan y optimiza  
✅ **Eficiencia:** Evita cálculos intermedios innecesarios  
✅ **Predicate Pushdown:** Filtra en origen (no lee datos innecesarios)  
✅ **Column Pruning:** Solo lee columnas usadas

---

### 💼 Caso de Uso: Pipeline ETL

```python
# 1. Transformaciones (lazy - instantáneas)
df_raw = spark.table("raw.ventas")
df_clean = df_raw.filter("ventas IS NOT NULL")
df_enrich = df_clean.withColumn("margen", col("ventas") - col("costo"))
df_agg = df_enrich.groupBy("sucursal").sum("margen")

# 2. Acción (ejecuta TODO el pipeline optimizado)
df_agg.write.mode("overwrite").saveAsTable("gold.margen_por_sucursal")
```

**Spark optimiza:**
1. Predicate pushdown: Filtra NULL en lectura
2. Column pruning: Solo lee columnas necesarias
3. Join reordering: Mejor orden de operaciones

**Resultado:** 10-100x más rápido que ejecución eager.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg
import warnings
warnings.filterwarnings('ignore')

print("🐌 LAZY EVALUATION: Transformaciones y Acciones")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • Diferencia entre transformaciones (lazy) y acciones (eager)")
print("  • Cómo Spark construye el DAG")
print("  • Catalyst Optimizer")
print("  • Cache y Persist")

print("\n📖 Transformaciones (lazy):")
print("  - df.select(...), df.filter(...), df.groupBy(...)")
print("  - df.join(...), df.orderBy(...), df.withColumn(...)")

print("\n📖 Acciones (eager):")
print("  - df.count(), df.show(), df.collect()")
print("  - df.write.parquet(...), df.toPandas()")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 🐌 Lazy evaluation con datos reales de Los Andes Market

### ⚡ Pipeline lazy real

Con `ventas_mensuales_mendoza_h3` podemos construir un pipeline de transformaciones y ver cómo Spark las evalúa perezosamente:

```python
# Todo esto es LAZY — no ejecuta nada
df1 = spark.table("pandito_ds.default.ventas_mensuales_mendoza_h3")
df2 = df1.filter(col("ventas") > 50000)
df3 = df2.select("sucursal_nombre", "zona", "ventas")
df4 = df3.groupBy("zona").sum("ventas")

# Solo cuando llamamos una ACCIÓN, Spark ejecuta todo el DAG
df4.show()  # ¡Ahora sí!
```

---

### 🧠 Catalyst Optimizer en acción

Spark optimiza el pipeline automáticamente:
1. **Predicate Pushdown:** el filtro `ventas > 50000` se aplica al leer (no después)
2. **Column Pruning:** solo lee `sucursal_nombre`, `zona`, `ventas` (ignora lat, lon, h3...)
3. **Join Reordering:** si hubiera joins, elige el orden más eficiente

---

### 💡 Demostración con datos reales
* Medir el tiempo de transformaciones (lazy) vs acción (eager)
* Ver el efecto de cache en múltiples acciones
* Comparar count() en cacheado vs no cacheado

In [0]:
from pyspark.sql.functions import col, sum as _sum, count, avg, desc
import time

print("🐌 LAZY EVALUATION CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df_spark is not None:
    print("\n1️⃣  DEMO: Transformaciones NO ejecutan (lazy)")
    print("-"*70)

    start = time.time()
    # 4 transformaciones encadenadas (lazy)
    df1 = df_spark.filter(col("ventas") > 50000)
    df2 = df1.select("sucursal_nombre", "zona", "ventas", "fecha")
    df3 = df2.withColumn("ventas_miles", col("ventas") / 1000)
    df4 = df3.groupBy("zona").agg(_sum("ventas").alias("ventas_totales"))
    lazy_time = time.time() - start

    print(f"\n   4 transformaciones encadenadas: {lazy_time:.4f}s")
    print("   ⚠️  ¡Instantáneo! Spark solo registró el plan (lazy)")

    print("\n" + "="*70)
    print("\n2️⃣  ACCIÓN: count() ejecuta TODO el pipeline")
    print("-"*70)

    start = time.time()
    n_result = df4.count()
    action_time = time.time() - start

    print(f"\n   count() = {n_result} (zonas con ventas > 50000)")
    print(f"   ⏱️  Tiempo de ejecución: {action_time:.3f}s")
    print("   ⚡ Aquí Spark ejecutó: filter → select → withColumn → groupBy → count")

    print("\n" + "="*70)
    print("\n3️⃣  OTRA ACCIÓN: show() ejecuta TODO DE NUEVO")
    print("-"*70)

    start = time.time()
    df4.orderBy(desc("ventas_totales")).show()
    show_time = time.time() - start

    print(f"\n   ⏱️  Tiempo de show(): {show_time:.3f}s")
    print("   ⚠️  Sin cache, Spark recalcula TODO desde cero")

    print("\n" + "="*70)
    print("\n4️⃣  CACHE: Calcular una vez, reusar")
    print("-"*70)

    # Sin cache: cada acción recalcula
    df_filtered = df_spark.filter(col("ventas") > 100000)

    start = time.time()
    df_filtered.count()
    t1_no_cache = time.time() - start

    start = time.time()
    df_filtered.count()
    t2_no_cache = time.time() - start

    print(f"\n   Sin cache:")
    print(f"      1er count(): {t1_no_cache:.3f}s")
    print(f"      2do count(): {t2_no_cache:.3f}s (recalcula)")

    # Con cache
    df_cached = df_spark.filter(col("ventas") > 100000).cache()

    start = time.time()
    df_cached.count()
    t1_cache = time.time() - start

    start = time.time()
    df_cached.count()
    t2_cache = time.time() - start

    start = time.time()
    df_cached.agg(avg("ventas").alias("promedio")).show()
    t3_cache = time.time() - start

    print(f"\n   Con cache:")
    print(f"      1er count(): {t1_cache:.3f}s (calcula + cachea)")
    print(f"      2do count(): {t2_cache:.3f}s (reusa cache)")
    print(f"      avg() + show(): {t3_cache:.3f}s (reusa cache)")
    print(f"\n   💡 cache() evita recalcular el mismo pipeline")

    print("\n" + "="*70)
    print("\n5️⃣  EXPLAIN: Ver el plan de ejecución")
    print("-"*70)

    print("\n   Plan de ejecución de df4 (explain):")
    df4.explain()
    print("\n   💡 'PushedFilters' muestra que Catalyst movió el filter al origen")
    print("   'Project' muestra columnas podadas (column pruning)")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del Notebook 11_03

### ✅ Lo que aprendiste

1. **Lazy Evaluation:**
   ```python
   # Transformaciones NO ejecutan, solo registran en el DAG
   df1 = spark.table("ventas")           # No ejecuta
   df2 = df1.filter("ventas > 1000")     # No ejecuta
   df3 = df2.select("sucursal", "ventas") # No ejecuta
   df3.count()                            # ¡AHORA ejecuta todo!
   ```

2. **Transformaciones (Lazy):**
   ```python
   # Crean un nuevo DataFrame, NO ejecutan
   df.select(...)
   df.filter(...)
   df.groupBy(...)
   df.join(...)
   df.orderBy(...)
   df.withColumn(...)
   ```

3. **Acciones (Eager):**
   ```python
   # Ejecutan TODO el DAG y devuelven resultados
   df.count()          # Cuenta filas
   df.show()           # Muestra filas
   df.collect()        # Trae TODO a memoria
   df.take(n)          # Trae n filas
   df.write.parquet()  # Escribe a disco
   df.toPandas()       # Convierte a Pandas
   ```

4. **Catalyst Optimizer:**
   - Predicate Pushdown: filtra en origen antes de leer
   - Column Pruning: solo lee columnas usadas
   - Join Reordering: elige el mejor orden de joins

5. **Cache y Persist:**
   ```python
   # Sin cache: recálcula en cada acción
   df2 = df1.filter(...)
   df2.count()  # Ejecuta todo
   df2.show()   # Ejecuta TODO OTRA VEZ
   
   # Con cache: calcula una vez, reusa
   df2 = df1.filter(...).cache()
   df2.count()  # Ejecuta y cachea
   df2.show()   # USA el cache (instantáneo)
   df2.unpersist()  # Libera memoria
   ```

---

### 🛠️ Guía rápida de Lazy Evaluation

**Caso 1: Pipeline ETL optimizado**
```python
# Transformaciones (lazy - instantáneas)
df_raw = spark.table("raw.ventas")
df_clean = df_raw.filter("ventas IS NOT NULL")
df_enrich = df_clean.withColumn("margen", col("ventas") - col("costo"))
df_agg = df_enrich.groupBy("sucursal").sum("margen")

# Acción (ejecuta TODO optimizado por Catalyst)
df_agg.write.mode("overwrite").saveAsTable("gold.margen_por_sucursal")
```

**Caso 2: Cachear DataFrame reutilizado**
```python
df_cached = spark.table("ventas").filter("año == 2024").cache()
df_cached.count()                    # Materializa y cachea
total = df_cached.agg(sum("ventas")).collect()[0][0]  # Usa cache
promedio = df_cached.agg(avg("ventas")).collect()[0][0]  # Usa cache
df_cached.unpersist()                # Liberar al terminar
```

**Caso 3: Evitar collect() en datos grandes**
```python
# MALO: collect() trae TODO al driver
result = spark.table("ventas_50gb").collect()  # 💥 OutOfMemory

# BUENO: agregar antes de traer
result = (spark.table("ventas_50gb")
    .groupBy("sucursal")
    .agg(sum("ventas").alias("total"))
    .collect())  # ✅ Solo 5 filas
```

**Caso 4: Verificar el plan de ejecución**
```python
# Ver qué hace Spark antes de ejecutar
df.explain()          # Plan lógico + físico
df.explain("formatted")  # Plan detallado con optimizaciones
```

**Caso 5: Mínimo de acciones por pipeline**
```python
# MALO: múltiples count() recalculan todo
df.filter(col("ventas") > 1000).count()  # Ejecuta TODO
df.filter(col("ventas") > 1000).show()  # Ejecuta TODO OTRA VEZ

# BUENO: cachear una vez, múltiples acciones
df_f = df.filter(col("ventas") > 1000).cache()
df_f.count()  # Ejecuta y cachea
df_f.show()   # Usa cache
df_f.unpersist()
```

---

### 🏆 Resumen del Módulo 11

**Aprendiste:**

1. **11_01 - PySpark Core SparkSession:** Arquitectura distribuida, SparkSession, Spark vs Pandas
2. **11_02 - Arquitectura Spark y DataFrames:** Particiones, StructType, esquema explícito, tipos de datos
3. **11_03 - Transformaciones Lazy y Acciones:** Lazy evaluation, DAG, Catalyst, cache/persist

**Habilidades adquiridas:**
* ✅ Distinguir transformaciones (lazy) de acciones (eager)
* ✅ Entender cómo Catalyst Optimizer mejora el plan de ejecución
* ✅ Usar cache/persist para evitar recálculos
* ✅ Evitar collect() en datasets grandes
* ✅ Construir pipelines ETL optimizados con Spark

---


<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🐌 ¡Módulo 11 Completado!</h3>
  <p><i>"Dominas PySpark Core: SparkSession, arquitectura distribuida y lazy evaluation. Ahora puedes construir pipelines que escalan de MB a TB."</i></p>
</div>